# SABR Sanity Checks and Extra Validation

This notebook collects model-level checks and component-level validation for the SABR replication scaffold. The goal is to go beyond the paper tables and show that the implementation behaves correctly in limiting cases, preserves key structural properties, and remains consistent with independent reference calculations.

The checks are organized from the simplest limiting cases to heavier off-paper validation runs. A few of the later cells are more computationally expensive because they use larger Monte Carlo samples or the PDE/FDM benchmark.

## Contents

1. Environment and imports
2. Helper functions
3. `nu = 0` should recover the exact CEV model
4. `beta = 1, nu = 0` should recover Black-Scholes / lognormal pricing
5. Algorithm 1 check: conditional integrated-variance sampler against target moments
6. Algorithm 3 check: exact CEV sampler against `pyfeng.Cev`
7. `rho = 0` off-paper benchmark against the PDE/FDM solver
8. Martingale sanity check across maturities
9. `|rho| = 1` Islah edge-case stability
10. Our method vs Islah: martingale gap in a stressed regime
11. Strike-shape and no-arbitrage checks on an off-paper parameter set
12. `beta -> 1` continuity check
13. Quick validation summary

In [ ]:
from pathlib import Path
import sys
import platform
import importlib
import math

import numpy as np
import pandas as pd
import pyfeng as pf
from IPython.display import display

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from sabr_replicate import (
    FDMConfig,
    MonteCarloConfig,
    SABRParams,
    case_table_3,
    conditional_integrated_variance_moments,
    european_call_price,
    finite_difference_call_prices,
    martingale_test,
    raw_moments_to_central_stats,
    repeated_pricing,
    run_full_validation,
    sample_cev_exact,
    sample_conditional_integrated_variance,
    simulate_terminal_forward,
    simulate_terminal_forward_islah,
)

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('Project root:', PROJECT_ROOT)
for pkg in ['numpy', 'pandas', 'scipy', 'pyfeng']:
    mod = importlib.import_module(pkg)
    print(f'{pkg}: {getattr(mod, "__version__", "version unavailable")}')

In [ ]:
def summarize_distribution(samples: np.ndarray) -> dict[str, float]:
    samples = np.asarray(samples, dtype=float)
    mean = float(np.mean(samples))
    centered = samples - mean
    var = float(np.mean(centered * centered))
    std = math.sqrt(max(var, 0.0))
    cv = std / mean if abs(mean) > 1e-16 else np.nan
    skewness = float(np.mean(centered ** 3) / max(std ** 3, 1e-16)) if std > 0.0 else np.nan
    ex_kurtosis = float(np.mean(centered ** 4) / max(var ** 2, 1e-16) - 3.0) if var > 0.0 else np.nan
    return {
        'mean': mean,
        'var': var,
        'std': std,
        'cv': cv,
        'skewness': skewness,
        'ex_kurtosis': ex_kurtosis,
    }


def comparison_frame(estimate: dict[str, float], reference: dict[str, float]) -> pd.DataFrame:
    rows = []
    for key, ref in reference.items():
        est = float(estimate[key])
        abs_error = est - float(ref)
        rel_error = abs_error / float(ref) if abs(ref) > 1e-16 else np.nan
        rows.append(
            {
                'metric': key,
                'estimate': est,
                'reference': float(ref),
                'abs_error': abs_error,
                'rel_error': rel_error,
            }
        )
    return pd.DataFrame(rows)


def is_nonincreasing(values: np.ndarray, tol: float = 1e-12) -> bool:
    arr = np.asarray(values, dtype=float)
    return bool(np.all(np.diff(arr) <= tol))


def is_discrete_convex(values: np.ndarray, tol: float = 1e-12) -> bool:
    arr = np.asarray(values, dtype=float)
    if arr.size < 3:
        return True
    second_diff = arr[:-2] - 2.0 * arr[1:-1] + arr[2:]
    return bool(np.all(second_diff >= -tol))

## 1. `nu = 0` should recover the exact CEV model

When `nu = 0`, the volatility path is deterministic and SABR reduces to the CEV model. This is the cleanest basic limit check for the `0 < beta < 1` branch.

We compare Monte Carlo prices from the SABR simulator with the exact CEV prices from `pyfeng.Cev`.

In [ ]:
params = SABRParams(f0=1.0, sigma0=0.25, nu=0.0, rho=-0.4, beta=0.3)
mc = MonteCarloConfig(maturity=1.0, step=1.0, n_paths=200_000, seed=12345)
strikes = np.array([0.6, 1.0, 1.4])

terminal = simulate_terminal_forward(params, mc)
mc_prices = np.array([european_call_price(terminal, float(k)) for k in strikes])
cev_prices = np.asarray(
    pf.Cev(sigma=params.sigma0, beta=params.beta, is_fwd=True).price(strikes, params.f0, mc.maturity),
    dtype=float,
)

pd.DataFrame(
    {
        'strike': strikes,
        'mc_price': mc_prices,
        'cev_price': cev_prices,
        'error': mc_prices - cev_prices,
    }
)

## 2. `beta = 1, nu = 0` should recover Black-Scholes / lognormal pricing

When `beta = 1` and `nu = 0`, the SABR dynamics reduce to the standard lognormal model. This checks the `beta = 1` special branch against `pyfeng.Bsm`.

In [ ]:
params = SABRParams(f0=1.0, sigma0=0.2, nu=0.0, rho=-0.75, beta=1.0)
mc = MonteCarloConfig(maturity=1.0, step=1.0, n_paths=200_000, seed=12345)
strikes = np.array([0.8, 1.0, 1.2])

terminal = simulate_terminal_forward(params, mc)
mc_prices = np.array([european_call_price(terminal, float(k)) for k in strikes])
bsm_prices = np.asarray(
    pf.Bsm(sigma=params.sigma0, is_fwd=True).price(strikes, params.f0, mc.maturity),
    dtype=float,
)

pd.DataFrame(
    {
        'strike': strikes,
        'mc_price': mc_prices,
        'bsm_price': bsm_prices,
        'error': mc_prices - bsm_prices,
    }
)

## 3. Algorithm 1 check: conditional integrated-variance sampler against target moments

The first nontrivial numerical component in the scheme is the conditional sampling of the normalized integrated variance `I_t^h`. Here we freeze a single conditional configuration `(sigma_t, sigma_{t+h}, nu, h)`, draw many samples from the sampler, and compare the sample moments with the target conditional moments returned by the analytical formulas exposed through PyFENG.

Because the implementation uses the fixed-shift SLN approximation, the mean and variance should line up most closely, while skewness and ex-kurtosis are expected to be close but not exact.

In [ ]:
sigma_t = np.full(250_000, 0.20)
sigma_next = np.full(250_000, 0.24)
nu = 0.4
h = 1.0
rng = np.random.default_rng(12345)

ih_samples = sample_conditional_integrated_variance(sigma_t, sigma_next, nu, h, rng)
mu1, mu2_raw, mu3_raw, mu4_raw = conditional_integrated_variance_moments(sigma_t[:1], sigma_next[:1], nu, h)
mean_t, var_t, std_t, cv_t, skew_t, exk_t = raw_moments_to_central_stats(mu1, mu2_raw, mu3_raw, mu4_raw)

target_stats = {
    'mean': float(mean_t[0]),
    'var': float(var_t[0]),
    'cv': float(cv_t[0]),
    'skewness': float(skew_t[0]),
    'ex_kurtosis': float(exk_t[0]),
}
sample_stats = summarize_distribution(ih_samples)

comparison_frame(
    {key: sample_stats[key] for key in target_stats},
    target_stats,
)

## 4. Algorithm 3 check: exact CEV sampler against `pyfeng.Cev`

The exact CEV transition sampler is also important enough to test on its own, without the rest of the SABR machinery around it. We therefore call `sample_cev_exact` directly and compare option prices from its samples with exact CEV prices from `pyfeng.Cev`.

This isolates the transition sampler from the conditional-volatility and integrated-variance layers.

In [ ]:
beta = 0.4
f_bar = 1.0
maturity = 1.0
sigma_cev = 0.35
variance_scale = sigma_cev ** 2 * maturity
n_samples = 250_000
rng = np.random.default_rng(23456)
strikes = np.array([0.6, 0.8, 1.0, 1.2, 1.4])

samples = sample_cev_exact(
    np.full(n_samples, f_bar),
    np.full(n_samples, variance_scale),
    beta,
    rng,
)
mc_prices = np.array([european_call_price(samples, float(k)) for k in strikes])
cev_prices = np.asarray(pf.Cev(sigma=sigma_cev, beta=beta, is_fwd=True).price(strikes, f_bar, maturity), dtype=float)

pd.DataFrame(
    {
        'strike': strikes,
        'mc_price': mc_prices,
        'cev_price': cev_prices,
        'error': mc_prices - cev_prices,
    }
)

## 5. `rho = 0` off-paper benchmark against the PDE/FDM solver

The paper tables are not the only place where the scheme should behave sensibly. Here we choose an off-paper parameter set with `rho = 0` and compare Monte Carlo prices with the built-in PDE/FDM benchmark.

This is useful for two reasons. First, `rho = 0` is a special SABR case with a simpler structure. Second, the parameter set is outside the paper's named examples, so this is a genuine sample-out validation run.

In [ ]:
params = SABRParams(f0=0.95, sigma0=0.27, nu=0.45, rho=0.0, beta=0.55)
maturity = 2.5
strikes = np.array([0.6, 0.8, 0.95, 1.1, 1.3])
fdm_config = FDMConfig(n_f=160, n_y=72, n_t_per_year=200, min_n_t=200)

fdm_frame = finite_difference_call_prices(params, maturity, strikes, config=fdm_config)
benchmark = {float(row.strike): float(row.fdm_price) for row in fdm_frame.itertuples(index=False)}

rho0_summary = repeated_pricing(
    params=params,
    mc=MonteCarloConfig(maturity=maturity, step=0.25, n_paths=80_000, seed=12345),
    strikes=strikes,
    n_repeats=8,
    seed0=12345,
    benchmark_prices=benchmark,
)

rho0_summary.assign(relative_error_pct=100.0 * rho0_summary['relative_error'])

## 6. Martingale sanity check across maturities

A core claim of the paper's conditional-forward construction is that it preserves the martingale property more reliably than the Islah approximation. Before comparing methods directly, it is useful to check that the main implementation does not show obvious systematic drift across maturities.

We reuse Case V because it is one of the more challenging paper parameter sets.

In [ ]:
case_v = case_table_3()['Case V']
params = SABRParams(
    f0=case_v['f0'],
    sigma0=case_v['sigma0'],
    nu=case_v['nu'],
    rho=case_v['rho'],
    beta=case_v['beta'],
)

martingale_test(params, maturities=[1, 2, 4, 6, 8, 10], step=1.0, n_paths=30_000, seed0=101)

## 7. `|rho| = 1` Islah edge-case stability

This is the edge case that used to sit only in a regression test. The notebook version makes the result visible. The question is not whether Islah is the preferred approximation, but whether the branch is numerically stable and non-pathological when `|rho| = 1`.

In [ ]:
rows = []
for beta in (0.4, 0.6, 0.8):
    params = SABRParams(f0=1.0, sigma0=0.2, nu=0.2, rho=1.0, beta=beta)
    mc = MonteCarloConfig(maturity=1.0, step=1.0, n_paths=5_000, seed=123)
    terminal = simulate_terminal_forward_islah(params, mc)
    rows.append(
        {
            'beta': beta,
            'all_finite': bool(np.isfinite(terminal).all()),
            'all_nonnegative': bool((terminal >= 0.0).all()),
            'mean_terminal': float(np.mean(terminal)),
            'min_terminal': float(np.min(terminal)),
        }
    )

pd.DataFrame(rows)

## 8. Our method vs Islah: martingale gap in a stressed regime

The paper argues that the CEV-based conditional-forward approximation is preferable in part because it preserves the martingale property more naturally than Islah's approximation. To make that visible, we compare the two schemes on a stressed parameter set with high correlation, nontrivial vol-of-vol, and multiple maturities.

The exact numbers will move with the random seed, but the comparison is still useful as a structural check.

In [ ]:
stress_params = SABRParams(f0=1.0, sigma0=0.35, nu=0.7, rho=0.9, beta=0.35)
maturities = [1, 2, 4, 6, 8, 10]

our_df = martingale_test(
    stress_params,
    maturities=maturities,
    step=0.5,
    n_paths=50_000,
    seed0=700,
    simulator=simulate_terminal_forward,
)
islah_df = martingale_test(
    stress_params,
    maturities=maturities,
    step=0.5,
    n_paths=50_000,
    seed0=700,
    simulator=simulate_terminal_forward_islah,
)

comparison = pd.DataFrame(
    {
        'maturity': maturities,
        'our_martingale_error': our_df['martingale_error'].to_numpy(),
        'islah_martingale_error': islah_df['martingale_error'].to_numpy(),
        'our_abs_z': our_df['z_score'].abs().to_numpy(),
        'islah_abs_z': islah_df['z_score'].abs().to_numpy(),
    }
)
summary = pd.DataFrame(
    [
        {
            'model': 'Our method',
            'mean_abs_error': float(our_df['martingale_error'].abs().mean()),
            'max_abs_error': float(our_df['martingale_error'].abs().max()),
            'mean_abs_z': float(our_df['z_score'].abs().mean()),
            'max_abs_z': float(our_df['z_score'].abs().max()),
        },
        {
            'model': 'Islah',
            'mean_abs_error': float(islah_df['martingale_error'].abs().mean()),
            'max_abs_error': float(islah_df['martingale_error'].abs().max()),
            'mean_abs_z': float(islah_df['z_score'].abs().mean()),
            'max_abs_z': float(islah_df['z_score'].abs().max()),
        },
    ]
)

display(summary)
comparison

## 9. Strike-shape and no-arbitrage checks on an off-paper parameter set

A good sanity check is not only about matching benchmarks, but also about preserving basic option-price shape restrictions. For a fixed terminal sample, call prices should be nonnegative, bounded above by the forward, decreasing in strike, and convex in strike.

We test those properties on an off-paper parameter set using a single shared terminal sample.

In [ ]:
params = SABRParams(f0=1.0, sigma0=0.28, nu=0.45, rho=-0.35, beta=0.55)
mc = MonteCarloConfig(maturity=3.0, step=0.25, n_paths=150_000, seed=54321)
terminal = simulate_terminal_forward(params, mc)
strikes = np.linspace(0.5, 1.5, 11)
call_prices = np.array([european_call_price(terminal, float(k)) for k in strikes])
intrinsic = np.maximum(params.f0 - strikes, 0.0)
upper_bound = np.full_like(strikes, params.f0)

shape_summary = pd.Series(
    {
        'all_prices_above_intrinsic': bool(np.all(call_prices >= intrinsic - 1e-12)),
        'all_prices_below_forward': bool(np.all(call_prices <= upper_bound + 1e-12)),
        'monotone_in_strike': is_nonincreasing(call_prices),
        'discrete_convexity': is_discrete_convex(call_prices),
    }
)
shape_frame = pd.DataFrame(
    {
        'strike': strikes,
        'call_price': call_prices,
        'intrinsic_lower_bound': intrinsic,
        'forward_upper_bound': upper_bound,
    }
)

display(shape_summary.to_frame('result'))
shape_frame

## 10. `beta -> 1` continuity check

The code uses a separate special branch when `beta = 1`. A sensible implementation should behave smoothly as `beta` approaches `1` from below. This is not a formal proof, but it is a useful continuity check around the branch boundary.

We monitor the ATM call price and the mean terminal forward for a sequence of `beta` values approaching `1`.

In [ ]:
betas = [0.7, 0.85, 0.95, 0.99, 1.0]
rows = []
for beta in betas:
    params = SABRParams(f0=1.0, sigma0=0.25, nu=0.35, rho=-0.3, beta=beta)
    mc = MonteCarloConfig(maturity=1.0, step=0.25, n_paths=120_000, seed=2025)
    terminal = simulate_terminal_forward(params, mc)
    rows.append(
        {
            'beta': beta,
            'atm_call_price': european_call_price(terminal, params.f0),
            'mean_terminal': float(np.mean(terminal)),
            'std_terminal': float(np.std(terminal)),
        }
    )

pd.DataFrame(rows)

## 11. Quick validation summary

This final cell reruns the repository's quick validation harness. It is useful as a smoke test, but it should not be treated as the final replication verdict because the sample sizes are deliberately small.

For reporting-quality conclusions, the paper-scale runs remain the more relevant benchmark.

In [ ]:
validation = run_full_validation(quick_mode=True)
pd.Series(
    {
        'table1_status': validation['table1_status'],
        'table2_status': validation['table2_status'],
        'overall_conclusion': validation['overall_conclusion'],
        'replication_conclusion': validation['replication_conclusion'],
    }
)